---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [18]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = False
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: c:\Users\User\Documents\GitHub\echochamber-project-team-1
DeepSeek key: True
Gemini key: True
Model folosit: deepseek-chat
OK


## Corpus

In [19]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"c:\Users\User\Documents\GitHub\echochamber-project-team-1")

corpus_file = ROOT / "data" / "cleaned" / "corpus_youtube_sample.jsonl"

print(corpus_file.exists())
print(corpus_file)

corpus = pd.read_json(corpus_file, lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

True
c:\Users\User\Documents\GitHub\echochamber-project-team-1\data\cleaned\corpus_youtube_sample.jsonl
420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[RecorderRomania] Vă urmăresc pentru că sunteți cea mai bună și profesionistă sursă . Dar aici ați
[b1tvchannel] Despre Georgescu, nu cred că habaucii, analfabeții care, de fiecare dată când es
[happyfishteleviziune] Ce s-a intamplat din anul 598- 1798 ( Evul mediu intunecat ), acelasi lucru se v


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [20]:
# modifica dupa preferinte

AXA_1 = "epistemic"
AXA_2 = "geopolitic"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [21]:
AXA_1_DEFINITION = """
epistemic măsoară dacă textul exprimă scepticism sau neîncredere în cunoștințele științifice.
0 = absent
1 = prezent
2 = dominant
"""
AXA_2_DEFINITION = """
geopolitic măsoară dacă textul abordează aspecte geografice sau politice.
0 = absent
1 = prezent
2 = dominant
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [22]:
MINI_PROMPT = f"""
Ești un adnotator uman care analizează comentarii de pe YouTube pentru un proiect de cercetare. Fiecare comentariu trebuie adnotat pe baza a două axe tematice, alături de identificarea țintei politice, poziției față de aceasta și tonului general al textului.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2
DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}
REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON valid.
FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""
print(MINI_PROMPT)


Ești un adnotator uman care analizează comentarii de pe YouTube pentru un proiect de cercetare. Fiecare comentariu trebuie adnotat pe baza a două axe tematice, alături de identificarea țintei politice, poziției față de aceasta și tonului general al textului.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. epistemic
2. geopolitic
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
epistemic = 0 / 1 / 2
geopolitic = 0 / 1 / 2
DEFINIȚII:

epistemic măsoară dacă textul exprimă scepticism sau neîncredere în cunoștințele științifice.
0 = absent
1 = prezent
2 = dominant


geopolitic măsoară dacă textul abordează aspecte geografice sau politice.
0 = absent
1 = prezent
2 = dominant

REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, fol

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [23]:
TESTS = corpus.sample(20)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
386,yt_iH8jB4NlV9Y_Ugz2SGyAmn4oGKy_iql4AaABAg,georgesimionoficial,Episodul 2: Cum ne-au furat alegerile - Turism...,Violul de la tabara AUR nu este o minciuna! Ex...
332,yt_TpUDm4ay-B8_Ugwazimbs6v6VBK4OTB4AaABAg,RecorderRomania,DOCUMENTAR RECORDER. Justiție capturată,Este dacă este să respecte regulile și Adevăru...
77,yt_gv9JYLtI6bw_UgwjDm2Enix1WUdVmwZ4AaABAg,spotmediaro,Ce știu cetățenii despre poliția locală?,Trebuie desființarea miliție locala și jandarm...
0,yt_Vekhmz5OPCc_UgzpXOYhxlT3Nqo6h7J4AaABAg,NicusorDanRO,🟢 LIVE Declarații de presă susținute la Palatu...,jigodia aia de Georgescu nu lua intrebari inca...
358,yt_CY3LNhAfypM_UgxQm1cxGQHXax-15-N4AaABAg,georgesimionoficial,"Sâmbra Oilor, datini și tradiții românești. Du...",SIMION A MULȚUMIT CASEI ALBE PENTRU ANULAREA V...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [24]:
USE_GEMINI = False
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: deepseek-chat


In [25]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [27]:
results = []

for _, row in TESTS.iterrows():

    USER = f"""
CANAL:
{row.get("source_channel", "")}

TITLU VIDEO:
{row.get("video_title", "")}

COMENTARIU:
<<< {row["text"]} >>>
"""

    try:
        raw = llm(MINI_PROMPT, USER, max_tokens=300)

        print("=" * 80)
        print("ID:", row["id"])
        print()
        print("COMENTARIU:")
        print(row["text"])
        print()
        print("OUTPUT MODEL:")
        print(raw)

        results.append({
            "id": row["id"],
            "text": row["text"],
            "model_output": raw
        })

    except Exception as e:
        print("EROARE la id:", row["id"])
        print(e)

ID: yt_iH8jB4NlV9Y_Ugz2SGyAmn4oGKy_iql4AaABAg

COMENTARIU:
Violul de la tabara AUR nu este o minciuna! Exista raport medico-legal care a demonstrat faptul ca fata a fost victima unui viol. Sa nu recunosti intamplarea si sa o numesti minciuna arata doar cat de interesat e gunoiul asta de siguranta femeilor

OUTPUT MODEL:
{
  "target": "AUR",
  "stance": "anti",
  "tone": "acuzator",
  "epistemic": 0,
  "geopolitic": 0
}
ID: yt_TpUDm4ay-B8_Ugwazimbs6v6VBK4OTB4AaABAg

COMENTARIU:
Este dacă este să respecte regulile și Adevărul cinstit curat și fără diferente dintre cei dintre cetățeni și funcționari publici din România la nivelul national

OUTPUT MODEL:
{
  "target": "justiția din România",
  "stance": "pro",
  "tone": "mobilizator",
  "epistemic": 0,
  "geopolitic": 1
}
ID: yt_gv9JYLtI6bw_UgwjDm2Enix1WUdVmwZ4AaABAg

COMENTARIU:
Trebuie desființarea miliție locala și jandarmeriei!Ceaușescu avea doar miliția comunista și era deajuns acum e plin de putori card sug din bugetul local și națio

In [32]:
import json
import pandas as pd

typed_rows = []

for r in results:

    parsed = json.loads(r["model_output"])

    typed_rows.append({
        "text": r["text"],
        "target": parsed.get("target"),
        "stance": parsed.get("stance"),
        "tone": parsed.get("tone"),
        "epistemic": parsed.get("epistemic"),
        "geopolitic": parsed.get("geopolitic")
    })

typed = pd.DataFrame(typed_rows)

typed.head()

,text,target,stance,tone,epistemic,geopolitic
0,Violul de la tabara AUR nu este o minciuna! Ex...,AUR,anti,acuzator,0,0
1,Este dacă este să respecte regulile și Adevăru...,justiția din România,pro,mobilizator,0,1
2,Trebuie desființarea miliție locala și jandarm...,poliția locală și jandarmeria,anti,acuzator,0,1
3,jigodia aia de Georgescu nu lua intrebari inca...,Călin Georgescu,anti,acuzator,0,0
4,SIMION A MULȚUMIT CASEI ALBE PENTRU ANULAREA V...,George Simion,anti,acuzator,0,1


#  Minitipologie

In [33]:
def mini_typology(row):

    if row["epistemic"] >= 1 and row["geopolitic"] >= 1:
        return "comentariu politico-epistemic"

    elif row["epistemic"] >= 1:
        return "comentariu epistemic"

    elif row["geopolitic"] >= 1:
        return "comentariu geopolitic"

    else:
        return "comentariu fără axă dominantă"

In [34]:
typed["mini_typology"] = typed.apply(mini_typology, axis=1)

print(typed["mini_typology"])

0     comentariu fără axă dominantă
1             comentariu geopolitic
2             comentariu geopolitic
3     comentariu fără axă dominantă
4             comentariu geopolitic
5     comentariu fără axă dominantă
6             comentariu geopolitic
7             comentariu geopolitic
8              comentariu epistemic
9     comentariu fără axă dominantă
10            comentariu geopolitic
11    comentariu fără axă dominantă
12    comentariu fără axă dominantă
13    comentariu fără axă dominantă
14    comentariu fără axă dominantă
15    comentariu fără axă dominantă
16            comentariu geopolitic
17            comentariu geopolitic
18    comentariu fără axă dominantă
19            comentariu geopolitic
Name: mini_typology, dtype: str


## Interpretare mini-tipologie

Mini-tipologia construită pe baza axelor „epistemic” și „geopolitic” arată că majoritatea comentariilor din eșantion sunt de tip geopolitic sau fără axă dominantă. Comentariile geopolitice includ referințe la instituții, stat, politică sau contexte naționale și internaționale. Comentariile fără axă dominantă nu conțin elemente clare de scepticism epistemic sau referințe geopolitice. Axa epistemică apare mai rar în eșantion, ceea ce sugerează că majoritatea comentariilor analizate sunt mai degrabă orientate politic decât spre contestarea cunoașterii sau adevărului.

## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:
- Ce două axe ai ales?
Am ales axele „epistemic” și „geopolitic”.
- De ce le-ai ales? 
Am ales aceste axe deoarece permit identificarea comentariilor care exprimă scepticism, afirmații considerate adevărate sau referințe politice și internaționale. Le-am ales pentru că sunt relevante în analiza discursului politic și a polarizării din comentariile YouTube.
- Modelul a returnat JSON corect?
 Modelul a returnat în general JSON corect și coerent, conform structurii cerute în prompt..
- Care a fost cea mai mare problemă?
 Cea mai mare problemă a fost limita de consum a API-ului Gemini, care a produs eroarea RateLimitError și a întrerupt procesarea completă a eșantionului. Ulterior am recurs la utilizarea deepseek.
- Ce ai schimba în prompt?
 Aș îmbunătăți promptul prin instrucțiuni mai stricte pentru returnarea exclusivă a obiectului JSON și prin exemple suplimentare pentru valorile axelor.